In [13]:
from pathlib import Path
import plotly.graph_objects as go
import pandas as pd
import plotly.express as px

In [14]:
PROJECT_ROOT = Path("..")

EVENTS_PATH = PROJECT_ROOT / "data/interim/dashboard_events.parquet"
FIGURES_DIR = PROJECT_ROOT / "reports/figures"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [15]:
events = pd.read_parquet(EVENTS_PATH)

events["timestamp"] = pd.to_datetime(events["timestamp"], format="mixed")
events["date"] = events["timestamp"].dt.floor("D")

events

,timestamp,user,method,endpoint,status,dashboard_id,path,browser,browser_platform,browser_version,date
0,2018-08-09 14:00:00.000000,b1fababf379ab3d8@project-a.com,GET,dashboards.view_dashboard,200,marketing-overview,/dashboards/marketing-overview,chrome,macos,67.0.3396.99,2018-08-09
1,2018-08-09 14:00:00.000000,b1fababf379ab3d8@project-a.com,GET,dashboards.view_dashboard,200,marketing-overview,/dashboards/marketing-overview,chrome,macos,67.0.3396.99,2018-08-09
2,2018-08-09 14:00:00.000000,b1fababf379ab3d8@project-a.com,GET,dashboards.view_dashboard,200,marketing-overview,/dashboards/marketing-overview,chrome,macos,67.0.3396.99,2018-08-09
3,2018-08-09 14:00:00.000000,b1fababf379ab3d8@project-a.com,GET,dashboards.view_dashboard,200,marketing-overview,/dashboards/marketing-overview,chrome,macos,67.0.3396.99,2018-08-09
4,2018-08-09 14:00:00.000000,b1fababf379ab3d8@project-a.com,GET,dashboards.view_dashboard,200,marketing-overview,/dashboards/marketing-overview,chrome,macos,67.0.3396.99,2018-08-09
...,...,...,...,...,...,...,...,...,...,...,...
69147,2021-01-10 14:18:03.587892,9942c672b9354771@lampenwelt.de,GET,dashboards.view_dashboard,200,revenue-factura,/dashboards/revenue-factura,chrome,windows,87.0.4280.88,2021-01-10
69148,2021-01-10 14:19:02.411386,9942c672b9354771@lampenwelt.de,GET,dashboards.view_dashboard,200,sales-comparison,/dashboards/sales-comparison,chrome,windows,87.0.4280.88,2021-01-10
69149,2021-01-10 16:00:06.753419,91ec8c4f7243bd8d@lampenwelt.de,GET,dashboards.view_dashboard,200,weekly-sales-comparison,/dashboards/weekly-sales-comparison,chrome,windows,87.0.4280.88,2021-01-10
69150,2021-01-10 17:02:41.778970,27707832861149d0@lampenwelt.de,GET,dashboards.view_dashboard,200,sales-comparison,/dashboards/sales-comparison,chrome,macos,88.0.4324.50,2021-01-10


In [16]:
summary = {
    "rows": len(events),
    "date_min": events["timestamp"].min(),
    "date_max": events["timestamp"].max(),
    "unique_users": events["user"].nunique(),
    "unique_dashboards": events["dashboard_id"].nunique(),
    "total_views": len(events),
    "missing_timestamp": events["timestamp"].isna().sum(),
    "missing_user": events["user"].isna().sum(),
    "missing_dashboard_id": events["dashboard_id"].isna().sum(),
}

summary

{'rows': 69152,
 'date_min': Timestamp('2018-08-09 14:00:00'),
 'date_max': Timestamp('2021-01-10 17:04:16.000353'),
 'unique_users': 245,
 'unique_dashboards': 28,
 'total_views': 69152,
 'missing_timestamp': np.int64(0),
 'missing_user': np.int64(0),
 'missing_dashboard_id': np.int64(0)}

# Aggregations on daily level

In [17]:
daily_views = (
    events
    .groupby("date")
    .size()
    .reset_index(name="views")
    .sort_values("date")
)

full_dates = pd.DataFrame({
    "date": pd.date_range(
        start=daily_views["date"].min(),
        end=daily_views["date"].max(),
        freq="D"
    )
})

daily_views = (
    full_dates
    .merge(daily_views, on="date", how="left")
    .fillna({"views": 0})
)

daily_views["views"] = daily_views["views"].astype(int)

daily_views

,date,views
0,2018-08-09,30
1,2018-08-10,29
2,2018-08-11,0
3,2018-08-12,1
4,2018-08-13,31
...,...,...
881,2021-01-06,195
882,2021-01-07,149
883,2021-01-08,175
884,2021-01-09,13


# plot

In [18]:
fig = px.line(
    daily_views,
    x="date",
    y="views",
    title="Daily Dashboard Views",
    labels={
        "date": "Date",
        "views": "Dashboard views"
    },
)

fig.update_layout(
    template="plotly_white",
    height=500,
)

fig.show()


fig.write_image(FIGURES_DIR / "daily_views_trend.png", width=1000, height=600, scale=1)


# Seasonality

In [19]:
weekday_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"
]

weekly_usage = daily_views.copy()
weekly_usage["weekday"] = weekly_usage["date"].dt.day_name()

weekly_usage = (
    weekly_usage
    .groupby("weekday", as_index=False)["views"]
    .mean()
)

weekly_usage["weekday"] = pd.Categorical(
    weekly_usage["weekday"],
    categories=weekday_order,
    ordered=True
)

weekly_usage = weekly_usage.sort_values("weekday")

fig = px.bar(
    weekly_usage,
    x="weekday",
    y="views",
    title="Average Dashboard Views by Weekday",
    labels={
        "weekday": "Weekday",
        "views": "Average daily views"
    },
)

fig.update_layout(
    template="plotly_white",
    height=500,
)

fig.show()

fig.write_image(FIGURES_DIR / "weekly_seasonality.png", width=1000, height=600, scale=1)


In [20]:
monthly_usage = daily_views.copy()
monthly_usage["month"] = monthly_usage["date"].dt.month
monthly_usage["month_name"] = monthly_usage["date"].dt.month_name()

monthly_usage = (
    monthly_usage
    .groupby(["month", "month_name"], as_index=False)["views"]
    .mean()
    .sort_values("month")
)

fig = px.bar(
    monthly_usage,
    x="month_name",
    y="views",
    title="Average Dashboard Views by Month",
    labels={
        "month_name": "Month",
        "views": "Average daily views"
    },
)

fig.update_layout(
    template="plotly_white",
    height=500,
)

fig.show()

fig.write_image(FIGURES_DIR / "monthly_seasonality.png", width=1000, height=600, scale=1)


In [21]:
fig = px.histogram(
    daily_views,
    x="views",
    nbins=40,
    title="Distribution of Daily Dashboard Views",
    labels={
        "views": "Daily dashboard views"
    },
)

fig.update_layout(
    template="plotly_white",
    height=500,
)

fig.show()


fig.write_image(FIGURES_DIR / "daily_views_distribution.png", width=1000, height=600, scale=1)

daily_views["views"].describe()

count    886.000000
mean      78.049661
std       68.754651
min        0.000000
25%       13.000000
50%       62.000000
75%      135.000000
max      430.000000
Name: views, dtype: float64

In [22]:
daily_activity = (
    events
    .groupby("date")
    .agg(
        views=("dashboard_id", "size"),
        active_dashboards=("dashboard_id", "nunique"),
    )
    .reset_index()
)

print(daily_activity.head())
daily_activity[["views", "active_dashboards"]].corr()

        date  views  active_dashboards
0 2018-08-09     30                  2
1 2018-08-10     29                  4
2 2018-08-12      1                  1
3 2018-08-13     31                  5
4 2018-08-14     47                  4


,views,active_dashboards
views,1.000000,0.802506
active_dashboards,0.802506,1.000000


In [23]:
daily_views = (
    events
    .groupby("date")
    .size()
    .reset_index(name="views")
    .sort_values("date")
)

full_dates = pd.DataFrame({
    "date": pd.date_range(
        start=daily_views["date"].min(),
        end=daily_views["date"].max(),
        freq="D"
    )
})

daily_views = (
    full_dates
    .merge(daily_views, on="date", how="left")
    .fillna({"views": 0})
)

daily_views["views"] = daily_views["views"].astype(int)

daily_views["views_7d_avg"] = daily_views["views"].rolling(7, min_periods=1).mean()
daily_views["views_30d_avg"] = daily_views["views"].rolling(30, min_periods=1).mean()

daily_views.head()

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=daily_views["date"],
        y=daily_views["views"],
        mode="lines",
        name="Daily views",
        opacity=0.35,
    )
)

fig.add_trace(
    go.Scatter(
        x=daily_views["date"],
        y=daily_views["views_7d_avg"],
        mode="lines",
        name="7-day rolling average",
    )
)

fig.add_trace(
    go.Scatter(
        x=daily_views["date"],
        y=daily_views["views_30d_avg"],
        mode="lines",
        name="30-day rolling average",
    )
)

fig.update_layout(
    title="Daily Dashboard Views with Rolling Trend",
    template="plotly_white",
    height=550,
    xaxis_title="Date",
    yaxis_title="Dashboard views",
    hovermode="x unified",
)

fig.show()

# save as smaller image
fig.write_image(FIGURES_DIR / "trend_rolling_average.png", width=1000, height=600, scale=1)

# Maybe in the requests files i can find somehow when a dashobard is created. If there are more active dashboards, number of dashboard views will go up. It would take time, simple forecast is requred for now.